# Diagnostic plots from `polarbearings` — plot-ready data for Plotly

This notebook shows how to produce the data behind common model-evaluation plots
**using only the current `polarbearings` API**, and hands each result straight to
Plotly. The goal is to see *how clunky or clean* each one is today, so we can
decide which dedicated helpers are worth adding.

Covered: **ROC / AUC**, **precision–recall**, **cost curves** (per-cell costs
swept across thresholds), and **calibration** (already shipped as a helper).

> Run with the `notebooks` dependency group:
> `uv run --group notebooks jupyter lab notebooks/diagnostics.ipynb`

In [1]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio

import polarbearings as pb
from polarbearings.calibration import calibration_curve

pio.renderers.default = "notebook_connected"  # CDN plotly.js -> lighter file

rng = np.random.default_rng(0)
n = 4_000
y = rng.integers(0, 2, n)
score = 1 / (1 + np.exp(-rng.normal(y * 1.3, 1.0)))  # sigmoid logits, in [0, 1]
df = pl.DataFrame({"y": y, "score": score})
df.head()

y,score
i64,f64
1,0.944633
1,0.563155
1,0.655656
0,0.335535
0,0.686714


## The workhorse: `threshold_sweep(confusion_matrix, ...)`

Every threshold-based curve (ROC, PR, cost) is a function of the four confusion
cells `tp, fp, fn, tn` across a grid of thresholds. `threshold_sweep` evaluates
`confusion_matrix` at every threshold in a **single pass**, and because each
confusion struct carries its own `threshold`, the wide result collapses to a tidy
long frame with `concat_list(...).explode(...).unnest(...)` — no name-parsing, no
parallel threshold bookkeeping. One sweep feeds every plot below.

In [2]:
from polarbearings import confusion_matrix, threshold_sweep

def confusion_sweep(frame, target, score, thresholds, weight=None):
    """Every threshold's confusion cells in ONE pass; the threshold rides in the struct."""
    exprs = threshold_sweep(confusion_matrix, target, score, thresholds, weight=weight)
    return (
        frame.select(pl.concat_list(exprs).alias("cm"))
        .explode("cm")
        .unnest("cm")  # -> threshold, tp, fp, fn, tn
    )

thresholds = np.linspace(0.0, 1.0, 101)
sweep = confusion_sweep(df, "y", "score", list(thresholds))

curve = sweep.with_columns(
    tpr=pl.col("tp") / (pl.col("tp") + pl.col("fn")),
    fpr=pl.col("fp") / (pl.col("fp") + pl.col("tn")),
    precision=pl.when(pl.col("tp") + pl.col("fp") > 0)
    .then(pl.col("tp") / (pl.col("tp") + pl.col("fp")))
    .otherwise(1.0),
    recall=pl.col("tp") / (pl.col("tp") + pl.col("fn")),
)
curve.select("threshold", "tp", "fp", "fn", "tn", "tpr", "fpr", "precision", "recall").head()

threshold,tp,fp,fn,tn,tpr,fpr,precision,recall
f64,i64,i64,i64,i64,f64,f64,f64,f64
0.0,2029,1971,0,0,1.0,1.0,0.50725,1.0
0.01,2029,1971,0,0,1.0,1.0,0.50725,1.0
0.02,2029,1971,0,0,1.0,1.0,0.50725,1.0
0.03,2029,1971,0,0,1.0,1.0,0.50725,1.0
0.04,2029,1971,0,0,1.0,1.0,0.50725,1.0


## ROC / AUC

The scalar AUC is a one-liner (`pb.roc_auc`). The **curve** is derived from the
sweep above: `fpr` vs `tpr`. `polarbearings` has no `roc_curve` helper today, so
we reshape the sweep ourselves.

In [3]:
auc = df.select(pb.roc_auc("y", "score")).to_series()[0]
roc = curve.sort("fpr")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=roc["fpr"].to_list(), y=roc["tpr"].to_list(), mode="lines", name="ROC",
    customdata=roc["threshold"].to_list(),
    hovertemplate="FPR=%{x:.3f}<br>TPR=%{y:.3f}<br>thr=%{customdata:.2f}<extra></extra>",
))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(title=f"ROC curve (AUC = {auc:.3f})", xaxis_title="False positive rate",
                  yaxis_title="True positive rate", width=560, height=480)
fig

**Now shipped as `confusion_curve`.** `threshold_sweep` gives the curve on a grid
in one pass; for the *exact* curve — one vertex per distinct score via a single
sorted pass — use `confusion_curve` (see "Exact curves" below). It returns the same
`threshold, tp, fp, fn, tn` schema, so the ROC is identical column math.

## Precision–Recall

Same sweep, plotted as `recall` vs `precision`. The scalar summary is
`pb.average_precision`.

In [4]:
ap = df.select(pb.average_precision("y", "score")).to_series()[0]
pr = curve.sort("recall")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pr["recall"].to_list(), y=pr["precision"].to_list(), mode="lines", name="PR",
    customdata=pr["threshold"].to_list(),
    hovertemplate="recall=%{x:.3f}<br>precision=%{y:.3f}<br>thr=%{customdata:.2f}<extra></extra>",
))
base = float((df["y"] == 1).mean())
fig.add_hline(y=base, line_dash="dash", line_color="gray",
              annotation_text=f"baseline = {base:.2f}")
fig.update_layout(title=f"Precision-Recall (AP = {ap:.3f})", xaxis_title="Recall",
                  yaxis_title="Precision", width=560, height=480)
fig

**Helper verdict — same as ROC:** nice-to-have. `pr_curve(target, score)` would
give exact vertices and tidy `(threshold, precision, recall)` output, but the data
is already one `threshold_sweep` pass away.

## Cost curves

A cost curve applies a **cost (or benefit) to each confusion cell** and sweeps the
total across thresholds — exactly the cells the sweep already has. Here: a false
negative costs 5, a false positive costs 1, correct calls are free. The optimal
operating threshold is the argmin.

In [5]:
costs = {"tp": 0.0, "fp": 1.0, "fn": 5.0, "tn": 0.0}
cost = sweep.with_columns(
    total_cost=(
        costs["tp"] * pl.col("tp") + costs["fp"] * pl.col("fp")
        + costs["fn"] * pl.col("fn") + costs["tn"] * pl.col("tn")
    )
)
best = cost.sort("total_cost").row(0, named=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=cost["threshold"].to_list(), y=cost["total_cost"].to_list(),
                         mode="lines", name="total cost"))
fig.add_vline(x=best["threshold"], line_dash="dash", line_color="crimson",
              annotation_text=f"min @ {best['threshold']:.2f}")
fig.update_layout(title="Total cost vs threshold (FN=5, FP=1)", xaxis_title="threshold",
                  yaxis_title="total cost", width=620, height=420)
fig

In [6]:
# The whole cost curve is one expression over the swept cells — trivially
# extended to expected $ per decision, utility, net benefit, etc.
print(f"min-cost threshold = {best['threshold']:.2f}  (total cost = {best['total_cost']:,.0f})")
cost.select("threshold", "fp", "fn", "total_cost").filter(
    pl.col("threshold").is_between(0.30, 0.45)
)

min-cost threshold = 0.34  (total cost = 1,661)


threshold,fp,fn,total_cost
f64,i64,i64,f64
0.3,1558,26,1688.0
0.31,1532,28,1672.0
0.32,1503,36,1683.0
0.33,1485,39,1680.0
0.34,1446,43,1661.0
…,…,…,…
0.41,1242,90,1692.0
0.42,1207,98,1697.0
0.43,1177,102,1687.0


**Helper verdict — optional sugar.** This is already a one-expression
derivation off `confusion_matrix`. A thin `expected_cost(costs, thresholds)` wrapper
(or a `by=` segmented version) would save typing, but there's no correctness or
performance gap — the primitive is enough.

## Calibration (already a helper)

`calibration_curve` ships today and returns plot-ready tidy data directly, with a
choice of binning strategy. Below: equal-width vs equal-frequency bins, overlaid on
the reliability diagonal.

In [7]:
uniform = calibration_curve(df, "y", "score", n_bins=10, strategy="uniform").collect()
quantile = calibration_curve(df, "y", "score", n_bins=10, strategy="quantile").collect()
uniform

bin,bin_lower,bin_upper,count,prob_pred,prob_true
i64,f64,f64,u32,f64,f64
0,0.0,0.1,21,0.078911,0.0
1,0.1,0.2,135,0.152685,0.02963
2,0.2,0.3,283,0.253508,0.077739
3,0.3,0.4,346,0.351849,0.16185
4,0.4,0.5,418,0.451818,0.222488
5,0.5,0.6,469,0.552241,0.373134
6,0.6,0.7,579,0.651577,0.525043
7,0.7,0.8,621,0.750779,0.673108
8,0.8,0.9,698,0.851921,0.80086


In [8]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                         line=dict(dash="dash", color="gray"), name="perfect"))
for name, data in [("uniform bins", uniform), ("quantile bins", quantile)]:
    fig.add_trace(go.Scatter(x=data["prob_pred"], y=data["prob_true"],
                             mode="lines+markers", name=name))
fig.update_layout(title="Calibration (reliability) curve",
                  xaxis_title="Mean predicted probability",
                  yaxis_title="Observed positive fraction",
                  width=560, height=480)
fig

In [9]:
# Custom edges (no scikit-learn equivalent): fixed, comparable score bands.
calibration_curve(df, "y", "score", bins=[0.0, 0.25, 0.5, 0.75, 1.0]).collect()

bin,bin_lower,bin_upper,count,prob_pred,prob_true
i64,f64,f64,u32,f64,f64
0,0.0,0.25,287,0.181285,0.041812
1,0.25,0.5,916,0.384916,0.177948
2,0.5,0.75,1372,0.635496,0.497813
3,0.75,1.0,1425,0.861058,0.821754


## Exact curves: `confusion_curve`

`confusion_curve` computes the confusion cells at **every distinct score** in one
sorted `O(n log n)` pass — the exact step function, not a grid — returning the same
`threshold, tp, fp, fn, tn` schema as the swept frame. Below, the exact ROC overlaid
on the 101-point grid from earlier.

In [10]:
from polarbearings import confusion_curve

exact = confusion_curve(df, "y", "score").with_columns(
    tpr=pl.col("tp") / (pl.col("tp") + pl.col("fn")),
    fpr=pl.col("fp") / (pl.col("fp") + pl.col("tn")),
).collect()

fig = go.Figure()
fig.add_trace(go.Scatter(x=exact["fpr"].to_list(), y=exact["tpr"].to_list(),
                         mode="lines", name=f"exact ({exact.height} pts)"))
fig.add_trace(go.Scatter(x=roc["fpr"].to_list(), y=roc["tpr"].to_list(),
                         mode="markers", name="grid (101 thresholds)", marker=dict(size=5)))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(title="Exact (confusion_curve) vs grid (threshold_sweep) ROC",
                  xaxis_title="False positive rate", yaxis_title="True positive rate",
                  width=560, height=480)
fig

## Summary

| Plot | Current API | Status |
|---|---|---|
| ROC / AUC | `roc_auc` + `threshold_sweep` (grid) or `confusion_curve` (exact) | Covered |
| Precision-Recall | `average_precision` + `confusion_curve` | Covered |
| Cost curve | one expression off the swept / curve cells | Optional — thin `expected_cost` wrapper |
| Calibration | `calibration_curve` (shipped) | Covered |

The threshold-based plots reduce to two primitives, both returning the same
`threshold, tp, fp, fn, tn` schema: `threshold_sweep(confusion_matrix, ...)` for a
fixed, comparable **grid** of operating points (great for monitoring), and
`confusion_curve` for the **exact** all-thresholds curve (one row per distinct
score, single sorted pass). `roc_curve`/`pr_curve`, if ever added, would just be
column-math wrappers over `confusion_curve`.